In [ ]:
import os
import re
import csv
import requests
from io import BytesIO
from pdfminer.high_level import extract_text
import pytesseract
import fitz  # PyMuPDF
import pandas as pd
from datetime import datetime
from PIL import Image
import PyPDF2
from rapidfuzz import process, fuzz
from IPython.display import display, HTML
import numpy as np

import re
import unicodedata
from pathlib import Path
from typing import Optional, Dict, Any, Tuple

In [27]:
df = pd.read_csv('elections_data.csv')
df.head()

,relation,title,date,type,format,language,identifier_urls,identifier_citation
0,https://dnpprepo.ub.rug.nl/88759/,Aanpassen aan de draagkracht van de aarde,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']",text,nl,['https://dnpprepo.ub.rug.nl/88759/1/DeGroenen...,De Groenen (2025) Aanpassen aan de draagkrach...
1,https://dnpprepo.ub.rug.nl/88739/,BBB levert,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']","['text', 'text']","['nl', 'nl']",['https://dnpprepo.ub.rug.nl/88739/7/BBB%20Ver...,BoerBurgerBeweging (2025) BBB levert. [Verki...
2,https://dnpprepo.ub.rug.nl/88765/,Bouwen op vertrouwen,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']","['text', 'text']","['nl', 'nl']",['https://dnpprepo.ub.rug.nl/88765/7/CDA%20Ver...,CDA (2025) Bouwen op vertrouwen. [Verkiezing...
3,https://dnpprepo.ub.rug.nl/88762/,De juiste aanpak voor Nederland,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']","['text', 'text']","['nl', 'nl']",['https://dnpprepo.ub.rug.nl/88762/7/JA21%20Ve...,JA21 (2025) De juiste aanpak voor Nederland. ...
4,https://dnpprepo.ub.rug.nl/88774/,"De mens centraal, niet het systeem",2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']",text,nl,['https://dnpprepo.ub.rug.nl/88774/1/Piratenpa...,"Piratenpartij (2025) De mens centraal, niet h..."


In [29]:
# change identifier_urls to string and only save first url
df['identifier_urls'] = df['identifier_urls'].apply(lambda x: str(x).split(',')[0] if pd.notna(x) else x)
# remove brackets and quotes from identifier_urls
df['identifier_urls'] = df['identifier_urls'].str.replace(r'[\[\]\'"]', '', regex=True)

In [30]:
df.head()

,relation,title,date,type,format,language,identifier_urls,identifier_citation
0,https://dnpprepo.ub.rug.nl/88759/,Aanpassen aan de draagkracht van de aarde,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']",text,nl,https://dnpprepo.ub.rug.nl/88759/1/DeGroenen%2...,De Groenen (2025) Aanpassen aan de draagkrach...
1,https://dnpprepo.ub.rug.nl/88739/,BBB levert,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']","['text', 'text']","['nl', 'nl']",https://dnpprepo.ub.rug.nl/88739/7/BBB%20Verki...,BoerBurgerBeweging (2025) BBB levert. [Verki...
2,https://dnpprepo.ub.rug.nl/88765/,Bouwen op vertrouwen,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']","['text', 'text']","['nl', 'nl']",https://dnpprepo.ub.rug.nl/88765/7/CDA%20Verki...,CDA (2025) Bouwen op vertrouwen. [Verkiezing...
3,https://dnpprepo.ub.rug.nl/88762/,De juiste aanpak voor Nederland,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']","['text', 'text']","['nl', 'nl']",https://dnpprepo.ub.rug.nl/88762/7/JA21%20Verk...,JA21 (2025) De juiste aanpak voor Nederland. ...
4,https://dnpprepo.ub.rug.nl/88774/,"De mens centraal, niet het systeem",2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']",text,nl,https://dnpprepo.ub.rug.nl/88774/1/Piratenpart...,"Piratenpartij (2025) De mens centraal, niet h..."


In [ ]:
def extract_text_from_pdf_url(url):
    # Download PDF
    response = requests.get(url)
    response.raise_for_status()

    # Load PDF into memory
    pdf_data = BytesIO(response.content)

    # Extract text
    text = extract_text(pdf_data)
    return text


# Example usage
url =  df['identifier_urls'].iloc[0]  # Pick first PDF URL
text = extract_text_from_pdf_url(url)

print(text[:1000])  # print first 1000 chars

Aanpassen aan 
de draagkracht 
van de aarde

DE GROENEN

partijprogramma 2025 v2

gewijzigd op 21 juni 2025 door het 95e partijcongres van De Groenen 

ISSN 2214-3742 € 0,=

partijprogramma De Groenen 2025 v2

A. De draagkracht van de aarde................................................................................... 3
1. De wetenschap als leidraad................................................................................ 3
2. Planetaire grenzen: aanpassen aan de draagkracht van de aarde.....................3
3. Brandstof, veeteelt en klimaat............................................................................ 4
4. Landbouw: herstel de biodiversiteit...................................................................6
5. Natuur: kwaliteit belangrijker dan kwantiteit....................................................8
6. Dierenrechten: respect voor de eigen waarde van het dier................................9
7. Biotechnologie: de EU gentechvrij........................

In [37]:
for idx, row in df.iterrows():
    pdf_path = row['identifier_urls']
    if pd.notna(pdf_path) and pdf_path.endswith('.pdf'):
        try:
            text = extract_text_from_pdf_url(pdf_path)
            df.at[idx, 'extracted_text'] = text
        except Exception as e:
            print(f"Error processing {pdf_path}: {e}")

The PDF <_io.BytesIO object at 0x000002A23A93EA20> contains a metadata field indicating that it should not allow text extraction. Ignoring this field and proceeding. Use the check_extractable if you want to raise an error in this case
The PDF <_io.BytesIO object at 0x000002A263104BD0> contains a metadata field indicating that it should not allow text extraction. Ignoring this field and proceeding. Use the check_extractable if you want to raise an error in this case
The PDF <_io.BytesIO object at 0x000002A2652B2F70> contains a metadata field indicating that it should not allow text extraction. Ignoring this field and proceeding. Use the check_extractable if you want to raise an error in this case


In [38]:
df.head()

,relation,title,date,type,format,language,identifier_urls,identifier_citation,extracted_text
0,https://dnpprepo.ub.rug.nl/88759/,Aanpassen aan de draagkracht van de aarde,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']",text,nl,https://dnpprepo.ub.rug.nl/88759/1/DeGroenen%2...,De Groenen (2025) Aanpassen aan de draagkrach...,Aanpassen aan \nde draagkracht \nvan de aarde\...
1,https://dnpprepo.ub.rug.nl/88739/,BBB levert,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']","['text', 'text']","['nl', 'nl']",https://dnpprepo.ub.rug.nl/88739/7/BBB%20Verki...,BoerBurgerBeweging (2025) BBB levert. [Verki...,Verkiezingsprogramma\n2025 - 2029\n\n Dankwoor...
2,https://dnpprepo.ub.rug.nl/88765/,Bouwen op vertrouwen,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']","['text', 'text']","['nl', 'nl']",https://dnpprepo.ub.rug.nl/88765/7/CDA%20Verki...,CDA (2025) Bouwen op vertrouwen. [Verkiezing...,Bouwen op\nvertrouwen.\n\nOnze keuzes voor\nee...
3,https://dnpprepo.ub.rug.nl/88762/,De juiste aanpak voor Nederland,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']","['text', 'text']","['nl', 'nl']",https://dnpprepo.ub.rug.nl/88762/7/JA21%20Verk...,JA21 (2025) De juiste aanpak voor Nederland. ...,eiligheid\n\n V\ne\nz\nn\nO\n\nOnze Gr\n\ne\nn...
4,https://dnpprepo.ub.rug.nl/88774/,"De mens centraal, niet het systeem",2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']",text,nl,https://dnpprepo.ub.rug.nl/88774/1/Piratenpart...,"Piratenpartij (2025) De mens centraal, niet h...",Verkiezingsprogramma Piratenpartij \nNederland...


In [ ]:


def clean_text(text: str) -> str:
    """Clean unwanted characters, including literal '\n', soft hyphens, ligatures."""
    if not isinstance(text, str):
        return text

    # Replace common PDF artifacts
    text = (text
            .replace('\xa0', ' ')   # non-breaking space
            .replace('\r', ' ')
            .replace(',', ' ')
            .replace('\u00ad', '')  # soft hyphen
            .replace('\ufb01', 'fi')# ﬁ ligature
            .replace('\ufb02', 'fl')# ﬂ ligature
           )

    # Remove both literal and real newlines/tabs -> single space
    text = re.sub(r'(\\n|\\r|\\t|\n|\r|\t)+', ' ', text)

    # Collapse multiple spaces and tidy punctuation spacing
    text = re.sub(r'\s{2,}', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    text = re.sub(r'\s+([,.;:!?])', r'\1', text)

    return text.strip()



In [44]:
# clean the extracted text in the dataframe
df['cleaned_text'] = df['extracted_text'].apply(clean_text)

In [40]:
# make new column 'party' by extracting party name (all characters before '(' in 'identifier_citation'extract_text_from_pdf_url
df['party'] = df['identifier_citation'].str.extract(r'^(.*?)\s*\(')[0].str.strip()
df.head()

,relation,title,date,type,format,language,identifier_urls,identifier_citation,extracted_text,party
0,https://dnpprepo.ub.rug.nl/88759/,Aanpassen aan de draagkracht van de aarde,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']",text,nl,https://dnpprepo.ub.rug.nl/88759/1/DeGroenen%2...,De Groenen (2025) Aanpassen aan de draagkrach...,Aanpassen aan \nde draagkracht \nvan de aarde\...,De Groenen
1,https://dnpprepo.ub.rug.nl/88739/,BBB levert,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']","['text', 'text']","['nl', 'nl']",https://dnpprepo.ub.rug.nl/88739/7/BBB%20Verki...,BoerBurgerBeweging (2025) BBB levert. [Verki...,Verkiezingsprogramma\n2025 - 2029\n\n Dankwoor...,BoerBurgerBeweging
2,https://dnpprepo.ub.rug.nl/88765/,Bouwen op vertrouwen,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']","['text', 'text']","['nl', 'nl']",https://dnpprepo.ub.rug.nl/88765/7/CDA%20Verki...,CDA (2025) Bouwen op vertrouwen. [Verkiezing...,Bouwen op\nvertrouwen.\n\nOnze keuzes voor\nee...,CDA
3,https://dnpprepo.ub.rug.nl/88762/,De juiste aanpak voor Nederland,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']","['text', 'text']","['nl', 'nl']",https://dnpprepo.ub.rug.nl/88762/7/JA21%20Verk...,JA21 (2025) De juiste aanpak voor Nederland. ...,eiligheid\n\n V\ne\nz\nn\nO\n\nOnze Gr\n\ne\nn...,JA21
4,https://dnpprepo.ub.rug.nl/88774/,"De mens centraal, niet het systeem",2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']",text,nl,https://dnpprepo.ub.rug.nl/88774/1/Piratenpart...,"Piratenpartij (2025) De mens centraal, niet h...",Verkiezingsprogramma Piratenpartij \nNederland...,Piratenpartij


In [45]:
df.head()

,relation,title,date,type,format,language,identifier_urls,identifier_citation,extracted_text,party,cleaned_text
0,https://dnpprepo.ub.rug.nl/88759/,Aanpassen aan de draagkracht van de aarde,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']",text,nl,https://dnpprepo.ub.rug.nl/88759/1/DeGroenen%2...,De Groenen (2025) Aanpassen aan de draagkrach...,Aanpassen aan \nde draagkracht \nvan de aarde\...,De Groenen,Aanpassen aan de draagkracht van de aarde DE G...
1,https://dnpprepo.ub.rug.nl/88739/,BBB levert,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']","['text', 'text']","['nl', 'nl']",https://dnpprepo.ub.rug.nl/88739/7/BBB%20Verki...,BoerBurgerBeweging (2025) BBB levert. [Verki...,Verkiezingsprogramma\n2025 - 2029\n\n Dankwoor...,BoerBurgerBeweging,Verkiezingsprogramma 2025 - 2029 Dankwoord Lie...
2,https://dnpprepo.ub.rug.nl/88765/,Bouwen op vertrouwen,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']","['text', 'text']","['nl', 'nl']",https://dnpprepo.ub.rug.nl/88765/7/CDA%20Verki...,CDA (2025) Bouwen op vertrouwen. [Verkiezing...,Bouwen op\nvertrouwen.\n\nOnze keuzes voor\nee...,CDA,Bouwen op vertrouwen. Onze keuzes voor een fat...
3,https://dnpprepo.ub.rug.nl/88762/,De juiste aanpak voor Nederland,2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']","['text', 'text']","['nl', 'nl']",https://dnpprepo.ub.rug.nl/88762/7/JA21%20Verk...,JA21 (2025) De juiste aanpak voor Nederland. ...,eiligheid\n\n V\ne\nz\nn\nO\n\nOnze Gr\n\ne\nn...,JA21,eiligheid V e z n O Onze Gr e n z e n Onze Wel...
4,https://dnpprepo.ub.rug.nl/88774/,"De mens centraal, niet het systeem",2025,"[""Verkiezingsprogramma's"", 'NonPeerReviewed']",text,nl,https://dnpprepo.ub.rug.nl/88774/1/Piratenpart...,"Piratenpartij (2025) De mens centraal, niet h...",Verkiezingsprogramma Piratenpartij \nNederland...,Piratenpartij,Verkiezingsprogramma Piratenpartij Nederland 2...


In [46]:
df.to_csv('elections_data_with_text.csv', index=False)